# 02 アルゴリズムの実装

- Algorithm A・B の**仕組み・入力と出力・前提条件**を説明する
- `.py` から import して実行し、**結果を確認・検証**する
- **計算量**を考える

アルゴリズムの本体は `algorithms/algorithm_a.py`・`algorithm_b.py` にあります（この Notebook には書き直していません）。

> 初期デモ（**線形探索 vs 二分探索**）は、アルゴリズムを比較する方法を示す**教材例**です。研究テーマを固定するものではありません。チームのテーマに合わせて、`algorithms/` と比較の課題（task）を置き換えてください。

## 準備：実験環境（最初に必ず実行）

F5 リポジトリの `.py`（`algorithms/`・`experiments/`）を import できるように、作業ディレクトリ（F5 のルート）を自動で探して設定します。

| 実行場所 | 準備 |
| --- | --- |
| ローカルの Jupyter / VSCode | `notebooks/` の中で開けば、そのまま動きます |
| Colab（方法A：GitHub から clone） | 下のセルの `REPO_URL` に**自分たちのリポジトリの URL** を入れる |
| Colab（方法B：ZIP をアップロード） | F5 フォルダを ZIP にして `/content` にアップロードしておく（またはセルの実行中にアップロード） |

Colab の CPU ランタイムで動きます（GPU・TPU は不要）。

In [ ]:
#@title 準備（F5 の .py を読み込む）
import os, sys, glob, zipfile, subprocess
from pathlib import Path

# ---- 方法A：GitHub から clone する場合だけ設定する（URL は自分たちのものを入れる）----
REPO_URL = ""                                # 例："https://github.com/<team>/<repo>.git"（空なら clone しない）
REPO_SUBDIR = "f5-open-research-project"     # リポジトリの中の F5 フォルダ（ルートが F5 なら ""）

IN_COLAB = "google.colab" in sys.modules


def is_f5_root(p):
    return (Path(p) / "algorithms" / "algorithm_a.py").exists() and (Path(p) / "experiments" / "benchmark.py").exists()


def find_f5_root():
    for p in [Path.cwd(), Path.cwd().parent]:          # ローカル：notebooks/ の中、または F5 のルート
        if is_f5_root(p):
            return p.resolve()
    if not IN_COLAB:
        return None
    if REPO_URL:                                        # Colab 方法A
        dest = Path("/content/repo")
        if not dest.exists():
            subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(dest)], check=True)
        cand = dest / REPO_SUBDIR if REPO_SUBDIR else dest
        if is_f5_root(cand):
            return cand
    zips = glob.glob("/content/*.zip")                  # Colab 方法B
    if not zips:
        from google.colab import files
        print("F5 フォルダの ZIP を選んでアップロードしてください")
        zips = ["/content/" + name for name in files.upload()]
    for z in zips:
        with zipfile.ZipFile(z) as zf:
            zf.extractall("/content/f5_upload")
    for hit in glob.glob("/content/f5_upload/**/algorithms/algorithm_a.py", recursive=True):
        root = Path(hit).parent.parent
        if is_f5_root(root):
            return root
    return None


F5_ROOT = find_f5_root()
if F5_ROOT is None:
    raise RuntimeError("F5 のフォルダが見つかりません。上の説明（方法A／方法B）を確認してください。")
os.chdir(F5_ROOT)
if str(F5_ROOT) not in sys.path:
    sys.path.insert(0, str(F5_ROOT))

# ---- 既存の .py（アルゴリズム・測定・比較）と、グラフ ----
import platform
import matplotlib.pyplot as plt
from algorithms import algorithm_a as alg_a     # TODO：研究テーマに合わせて置き換える
from algorithms import algorithm_b as alg_b
from experiments import benchmark as bm
from experiments import compare_algorithms as ca
from experiments import plots

plots.setup_style()                    # 文字を大きくするなら plots.setup_style(1.3)
SAVE_FIGURES = False                   # True にすると、グラフを notebooks/figures/ に PNG で保存する
FIG_DIR = "notebooks/figures"


def show(fig, name):
    """グラフを表示し、SAVE_FIGURES = True なら PNG も保存する"""
    if SAVE_FIGURES:
        print("saved:", plots.save(fig, f"{FIG_DIR}/{name}.png"))
    plt.show()


def table(rows, columns=ca.TABLE_COLUMNS):
    """比較結果の表。pandas があれば DataFrame、なければ文字の表（時間の列 _s はミリ秒）"""
    try:
        import pandas as pd
        from IPython.display import display
        data = [{h: (None if r.get(k) is None else (r[k] * 1e3 if k.endswith("_s") else r[k])) for k, h, _f in columns}
                for r in rows]
        display(pd.DataFrame(data))
    except ImportError:
        print(ca.format_table(rows, columns))


print("F5_ROOT:", F5_ROOT)
print("実行環境:", ca.environment(), "Colab" if IN_COLAB else "local")
print("Algorithm A:", alg_a.NAME, "/ Algorithm B:", alg_b.NAME)

## 1. Algorithm A と Algorithm B

| | Algorithm A：線形探索 | Algorithm B：二分探索 |
| --- | --- | --- |
| 仕組み | 先頭から 1 つずつ key と比べる | 範囲の中央と比べて、範囲を半分に絞り込む |
| 前提条件 | なし | **昇順にソート済み** |
| 前処理 | なし | ソート（O(N log N)） |
| 同じ値が複数あるとき | 最初の位置 | 最初の位置（下限探索。教科書の版とは異なる） |

**TODO**：チームのアルゴリズムの説明に書き換える

In [ ]:
for m in (alg_a, alg_b):
    print(f"{m.NAME}")
    print(f"  問題    : {m.PROBLEM}")
    print(f"  前提条件: {m.PRECONDITION}")
    print(f"  計算量  : {m.COMPLEXITY}")

## 2. 入力と出力・実行

In [ ]:
data = [42, 7, 19, 3, 88, 19, 55]
key = 19
idx_a = alg_a.linear_search(data, key)
sorted_data = alg_b.prepare(data)           # 前処理（ソート）。元の data は変わらない
idx_b = alg_b.binary_search(sorted_data, key)
print("入力 data       :", data, " key =", key)
print("A の出力        :", idx_a, "→ data[idx] =", data[idx_a])
print("前処理後のデータ:", sorted_data)
print("B の出力        :", idx_b, "→ sorted[idx] =", sorted_data[idx_b])
print("含まれない値    :", alg_a.linear_search(data, 100), alg_b.binary_search(sorted_data, 100))

## 3. 処理の様子（比較回数）

比較回数は**実行環境によらない**指標です。最悪の場合（含まれない値）で比べます。

In [ ]:
for n in [8, 64, 512, 4096]:
    a = list(range(n))
    _i, ca_ = alg_a.linear_search_count(a, -1)
    _j, cb_ = alg_b.binary_search_count(a, -1)
    print(f"N = {n:5}: 線形探索 {ca_:5} 回、二分探索 {cb_:3} 回")

## 4. 正しさの検証

- **同じ出力が期待される**場合：ソート済みの同じ配列なら、A と B は同じ添字（最初の位置）を返すはず
- **基準（オラクル）との照合**：Python の `in` で「含まれるか」を確かめる
- 空の入力・1 要素・先頭・末尾などの**境界条件**も確かめる（`tests/test_algorithms.py`）

In [ ]:
import random
rng = random.Random(0)
trials, mismatch = 1000, 0
for _ in range(trials):
    n = rng.randrange(0, 50)
    a = sorted(rng.randrange(0, 40) for _ in range(n))
    k = rng.randrange(-5, 45)
    ia, ib = alg_a.linear_search(a, k), alg_b.binary_search(a, k)
    if ia != ib or (ia != -1) != (k in a):
        mismatch += 1
print(f"ランダムな {trials} 通りの入力で、A・B・Python の in の答えが一致しなかった回数: {mismatch}")
print("境界条件:", alg_a.linear_search([], 1), alg_b.binary_search([], 1), alg_b.binary_search([5], 5))

In [ ]:
# unittest をこの Notebook から実行する（tests/ のテストがすべて成功するか）
import unittest
suite = unittest.defaultTestLoader.discover("tests")
result = unittest.TextTestRunner(verbosity=1).run(suite)
print("成功" if result.wasSuccessful() else "失敗したテストがあります")

## 5. 計算量の考察

| アルゴリズム | N の意味 | 最良 | 平均 | 最悪 | 空間 | 前処理 |
| --- | --- | --- | --- | --- | --- | --- |
| 線形探索 | データの要素数 | O(1) | O(N) | O(N) | O(1) | なし |
| 二分探索（この実装） | データの要素数 | O(log N) | O(log N) | O(log N) | O(1) | ソート O(N log N) |

- 二分探索は**ソート済みが前提**です。データを 1 回ソートして何回も探索するなら、ソートの時間は分け合えます（Notebook 03）。
- **TODO**：チームのアルゴリズムの計算量（最良・平均・最悪・空間）と、その理由を書く

## 6. 学生向け TODO

In [ ]:
# TODO 1：自分たちのアルゴリズムを algorithms/ に追加して import し、上と同じように実行・検証する
# 例：from algorithms import my_algorithm
# TODO 2：tests/ に、正しい結果・空の入力・境界条件・不正な入力のテストを追加する
# TODO 3：教科書のサンプル（textbook/README.md の一覧）を動かして、処理の流れを説明する
print(alg_a.linear_search_count([5, 3, 8], 8), alg_b.binary_search_count([3, 5, 8], 8))

## 考察（チームで記入）

- **TODO**：2 つのアルゴリズムの仕組みの違いを、自分の言葉で説明する
- **TODO**：前提条件（例：ソート済み）を満たさない入力を与えると、何が起きるか

## 参考文献

- 柴田望洋『新・明解 Pythonで学ぶアルゴリズムとデータ構造（第2版）』SBクリエイティブ（初期デモ：第3章 線形探索・2 分探索）
- F5 リポジトリの `README.md`（計算量、実行時間の測定、正しさの検証）、`textbook/README.md`（教科書コードの一覧と参照方法）
- TODO：チームが参考にした文献・データの出典を追加する